# BazaarBATNA — Seller Quality on Kaggle T4

End-to-end seller eval using **Gemma 4 E4B** (`google/gemma-4-E4B`).

**Setup before running:**
1. Settings → Accelerator → **GPU T4 x1** (or P100)
2. Settings → Internet → **On**
3. Add Kaggle Secret `HF_TOKEN` (with read access to gated models)
4. Model is Apache-2.0 (`google/gemma-4-E4B`) — no gating to accept

This notebook clones BazaarBATNA, downloads CraigslistBargains via Codalab,
runs `eval/seller_quality.py`, and prints the acceptance summary.


## 1 · Install deps

In [ ]:
!pip install -q --upgrade \
    "transformers>=4.46" "accelerate>=1.1" "bitsandbytes>=0.44" \
    "sentencepiece>=0.2" "datasets>=3.0" "huggingface_hub>=0.24" \
    "torch>=2.2" "requests>=2.31"


## 2 · HF auth

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HF login OK")


## 3 · Clone BazaarBATNA

Idempotent: wipes any prior clone so `Run All` always uses the latest main.


In [ ]:
import shutil, sys, subprocess, os

REPO_URL   = "https://github.com/paymybills/BazaarBATNA.git"
LOCAL_PATH = "/kaggle/working/metathon"

if os.path.exists(LOCAL_PATH):
    shutil.rmtree(LOCAL_PATH)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, LOCAL_PATH], check=True)

if LOCAL_PATH not in sys.path:
    sys.path.insert(0, LOCAL_PATH)
os.chdir(LOCAL_PATH)
print("Working dir:", os.getcwd())


## 4 · Download CraigslistBargains

HF dataset script for `craigslist_bargains` is deprecated. Pull parsed JSON
directly from Codalab (the original public bundle).


In [ ]:
import json, requests, time
from pathlib import Path

URLS = {
    "train": "https://worksheets.codalab.org/rest/bundles/0xd34bbbc5fb3b4fccbd19e10756ca8dd7/contents/blob/parsed.json",
    "dev":   "https://worksheets.codalab.org/rest/bundles/0x15c4160b43d44ee3a8386cca98da138c/contents/blob/parsed.json",
}

def _download(url: str, retries: int = 3):
    last = None
    for i in range(1, retries + 1):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e
            time.sleep(min(2 * i, 6))
    raise RuntimeError(f"download failed: {last}")

def _to_float(x):
    if x is None: return None
    if isinstance(x, (int, float)): return float(x)
    if isinstance(x, str):
        try: return float(x.replace("$", "").replace(",", "").strip())
        except ValueError: return None
    return None

def _flatten(example: dict) -> dict | None:
    sc = example.get("scenario") or {}
    kbs = sc.get("kbs") or []
    seller = next((kb for kb in kbs if (kb.get("personal") or {}).get("Role", "").lower() == "seller"), kbs[0] if kbs else None)
    if not seller:
        return None
    item = seller.get("item") or {}
    desc = item.get("Description")
    if isinstance(desc, list):
        desc = " ".join(str(x) for x in desc)
    price = _to_float(item.get("Price") or (seller.get("personal") or {}).get("Target"))
    if price is None:
        return None
    return {
        "category": sc.get("category", "unknown"),
        "title": str(item.get("Title") or "untitled"),
        "description": str(desc or ""),
        "price": price,
    }

Path("data").mkdir(exist_ok=True)
for split_name, out_name in [("train", "train"), ("dev", "dev")]:
    raw = _download(URLS[split_name])
    rows = [r for ex in raw if (r := _flatten(ex))]
    out = Path("data") / f"{out_name}.json"
    with out.open("w") as f:
        json.dump(rows, f)
    print(f"  {out}: {len(rows)} listings")


## 5 · Smoke test the seller

One open + one respond. Verifies the model loads and produces structured output.


In [ ]:
from data.craigslist_loader import sample_listing
from bazaarbot_env.llm_seller import LLMSeller

MODEL_ID = "google/gemma-4-E4B"

listing = sample_listing(seed=42)
brief = {
    "asking_price": float(listing["price"]),
    "reservation_price": round(float(listing["price"]) * 0.78, 2),
    "persona": "firm",
}

seller = LLMSeller(listing, brief, model=MODEL_ID)
print("Listing:", listing["title"], f"asking ${listing['price']:.0f}")
print()
print("OPEN:")
print("  ", seller.open())
print()
reply = seller.respond(
    history=[{"role": "seller", "message": "asking 100", "price": brief["asking_price"]}],
    buyer_message="I can do 60.",
    buyer_offer=60.0,
)
print("REPLY:", reply)


## 6 · Run seller-quality eval

50 episodes on the dev split, mixed personas. Logs to `runs/{ts}_seller_quality/`.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "eval/seller_quality.py",
    "--model", "google/gemma-4-E4B",
    "--split", "dev",
    "--n", "50",
    "--seed", "42",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 7 · Show the latest summary

In [ ]:
import json
from pathlib import Path

runs = sorted(Path("runs").glob("*_seller_quality"))
latest = runs[-1]
summary = json.loads((latest / "summary.json").read_text())
print(f"Run dir: {latest}\n")
print(json.dumps(summary, indent=2))


## 8 · Zip the run for download

Saves `/kaggle/working/seller_quality_run.zip`. After this cell finishes, the file
appears in the Kaggle notebook's right-hand "Output" panel for download.


In [ ]:
import shutil
from pathlib import Path

latest = sorted(Path("runs").glob("*_seller_quality"))[-1]
archive = shutil.make_archive("/kaggle/working/seller_quality_run", "zip", root_dir=latest)
print("Saved:", archive)
